# pipeline

> The headless decomposition pipeline: load a transcription run manifest, then per source
> per pipeline-segment run VAD + forced alignment, build one aligned segment per VAD chunk
> (times offset to source coordinates), commit the spine to the graph, and verify — with
> HITL approval seams between alignment, commit, and the next source.

In [ ]:
#| default_exp pipeline

In [ ]:
#| export
import json
import logging
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from cjm_plugin_system.core.manager import PluginManager
from cjm_plugin_system.core.queue import JobQueue, JobStatus
from cjm_plugin_system.core.ports import (
    Composition, CompositionNode, CompositionRun, NodeState, OutputRef,
)

# Typed wire-kind registration (stage 2): importing the DTO classes is what
# lets the proxy's wire_decode hand this host process TYPED results.
from cjm_media_plugin_system.core import MediaAnalysisResult
from cjm_forced_alignment_adapter_interface.core import ForcedAlignResult

from cjm_transcript_decomp_core.models import (
    DecompConfig, DecompSegment, DecompDocument, DecompManifest,
    FAWord, VADChunk, new_run_id,
)
from cjm_transcript_decomp_core.alignment import (
    map_fa_words_to_text, assign_words_to_chunks, build_segments_from_alignment,
    tier1_alignment_checks,
)
from cjm_transcript_decomp_core.graph import (
    build_graph_payload, commit_graph, verify_document,
)

logger = logging.getLogger(__name__)

In [ ]:
#| export
async def submit_and_wait(
    queue: JobQueue,   # Started job queue
    instance_id: str,  # Capability instance to invoke
    *,
    timeout: Optional[float] = None,  # Seconds to wait; None = no limit
    **kwargs,          # Forwarded to the capability's execute()
) -> Any:  # The completed job's result payload
    """Submit one capability job, wait for it, and return its result (raise on failure)."""
    job_id = await queue.submit(instance_id, **kwargs)
    job = await queue.wait_for_job(job_id, timeout=timeout)
    if job.status != JobStatus.completed:
        raise RuntimeError(f"{instance_id} job {job_id} {job.status}: {job.error}")
    return job.result

In [ ]:
#| export
def load_source_manifest(
    path: str,  # Path to a transcription-core run manifest JSON
) -> Dict[str, Any]:  # Parsed manifest dict
    """Load + lightly validate a transcription-core run manifest.

    Consumed as untyped JSON — the format/version tags are the only interchange
    contract (no shared schema type across cores; manifest-as-interchange, CR-20).
    """
    data = json.loads(Path(path).read_text())
    fmt = data.get("format", "")
    if "transcription-core" not in fmt:
        logger.warning(f"unexpected source manifest format: {fmt!r} (continuing)")
    return data

In [ ]:
#| export
def vad_chunks_from_result(
    result: MediaAnalysisResult,  # Typed VAD result (wire-decoded at the proxy)
) -> List[VADChunk]:  # Segment-local VAD chunks, sorted, re-indexed
    """Normalize a typed VAD result into segment-local VAD chunks.

    Pure normalizer (stage 3): the capability invocation moved into the
    per-source composition (`build_alignment_composition`); this folds one
    node's typed result. Per-segment VAD (not whole-source) is the validated
    decomp path — originally a browser Web-Audio constraint, kept on the
    merits (D11).
    """
    chunks = [VADChunk(index=0, start_time=float(r.start), end_time=float(r.end))
              for r in result.ranges]
    chunks.sort(key=lambda c: c.start_time)
    for i, c in enumerate(chunks):
        c.index = i
    return chunks

In [ ]:
#| export
def fa_words_from_result(
    result: "ForcedAlignResult",  # Typed FA result (wire-decoded at the proxy)
) -> List[FAWord]:  # Word-level alignment results
    """Normalize a typed forced-alignment result into FA words (pure; stage 3)."""
    return [FAWord(text=str(it.text), start_time=float(it.start_time),
                   end_time=float(it.end_time))
            for it in result.items]


def build_alignment_composition(
    seg_list: List[Dict[str, Any]],  # Pipeline-segment entries from the transcription manifest
    vad_id: str,          # VAD capability instance id
    fa_id: str,           # Forced-alignment capability instance id
    force: bool = False,  # Per-call cache-bypass control flag
) -> Tuple[Composition, List[Dict[str, Any]]]:  # (composition, per-pseg meta rows)
    """Build the whole-source M×(VAD ∥ FA) composition (the D8 fan-in shape).

    Each pipeline segment contributes two INDEPENDENT nodes consuming the
    same model-input WAV — VAD and FA fan in to the host's alignment fold.
    All inputs are host-known up-front (manifest data), so every kwarg is
    static; the win is dispatch: the CPU-resident VAD nodes parallelize
    under the queue's empirical admission while the GPU FA nodes serialize
    on headroom. Empty-text segments are recorded as skipped meta rows
    (the caller logs them) and contribute no nodes.
    """
    nodes: List[CompositionNode] = []
    metas: List[Dict[str, Any]] = []
    for i, pseg in enumerate(seg_list):
        # Manifest entries are JSON dicts (manifest-as-interchange, CR-20).
        model_input = str(pseg.get("model_input_path", ""))
        text = str(pseg.get("text", "") or "")
        seg_start = float(pseg.get("start", 0.0))
        job_id = pseg.get("job_id")
        if not text.strip():
            metas.append({"skipped": True, "seg_start": seg_start})
            continue
        vad_n, fa_n = f"vad_{i:04d}", f"fa_{i:04d}"
        nodes.append(CompositionNode(vad_n, vad_id,
                                     {"media_path": model_input, "force": force}))
        nodes.append(CompositionNode(fa_n, fa_id,
                                     {"audio": model_input, "text": text, "force": force}))
        metas.append({"skipped": False, "text": text, "seg_start": seg_start,
                      "job_id": job_id, "vad_node": vad_n, "fa_node": fa_n})
    return Composition(nodes=nodes), metas

In [ ]:
#| export
async def decompose_source(
    queue: JobQueue,
    cfg: DecompConfig,         # Run configuration
    source: Dict[str, Any],    # One source entry from the transcription manifest
    manifest_path: str,        # Consumed manifest path (for provenance)
    source_index: int,         # Position of this source within the run
    transcriber_name: str,     # Upstream transcriber capability name
) -> Tuple[str, List[DecompSegment]]:  # (source_path, ordered aligned segments)
    """Decompose one source's transcription segments into aligned graph segments.

    Stage 3 (CR-16 ports): the per-segment VAD ∥ FA invocations run as ONE
    whole-source composition — the queue's admission parallelizes the
    CPU-resident VAD nodes while the GPU FA nodes serialize on headroom —
    then the pure alignment fold runs per pipeline segment over the
    collected typed results, exactly as before (host logic stays in the
    core). Times are offset to source coordinates; indices accumulate across
    the source's pipeline segments.
    """
    source_path = str(source.get("source_path", ""))
    seg_list = list(source.get("segments") or [])

    comp, metas = build_alignment_composition(
        seg_list, cfg.vad_plugin, cfg.fa_plugin, force=cfg.force)
    results: Dict[str, Any] = {}
    if comp.nodes:
        comp_id = await queue.submit_composition(comp)
        crun = await queue.wait_for_composition(comp_id)
        if crun.status != NodeState.completed:
            failed = {nid: str(nr.error) for nid, nr in crun.node_runs.items()
                      if nr.state == NodeState.failed}
            raise RuntimeError(
                f"alignment composition {crun.status.value} for {source_path}: {failed}")
        results = crun.results_by_node()

    aligned: List[DecompSegment] = []
    global_index = 0
    for m in metas:
        seg_start = m["seg_start"]
        if m["skipped"]:
            logger.warning(f"[src {source_index}] empty text at {seg_start:.1f}s; "
                           f"skipping pipeline segment")
            continue
        text = m["text"]
        job_id = m["job_id"]
        vad_chunks = vad_chunks_from_result(results[m["vad_node"]])
        fa_words = fa_words_from_result(results[m["fa_node"]])
        spans = map_fa_words_to_text(text, fa_words)
        assignments = assign_words_to_chunks(fa_words, vad_chunks)
        text_segments = build_segments_from_alignment(
            text=text, spans=spans, assignments=assignments,
            num_chunks=len(vad_chunks), source_id=job_id, source_provider_id=transcriber_name,
        )
        for w in tier1_alignment_checks(text_segments, vad_chunks):
            logger.warning(f"[src {source_index}] pseg @ {seg_start:.1f}s: {w}")

        for ts, vc in zip(text_segments, vad_chunks):
            aligned.append(DecompSegment(
                index=global_index, text=ts.text,
                start_time=seg_start + vc.start_time, end_time=seg_start + vc.end_time,
                start_char=ts.start_char, end_char=ts.end_char,
                source_job_id=job_id, source_provider_id=transcriber_name,
                vad_chunk_index=vc.index,
            ))
            global_index += 1
        logger.info(f"[src {source_index}] pseg @ {seg_start:.1f}s -> "
                    f"{len(vad_chunks)} chunks, {len(fa_words)} words")
    return source_path, aligned

In [ ]:
#| export
def confirm_seam(
    seam: str,                 # Seam label, e.g. "alignment-review"
    summary_lines: List[str],  # What the operator is being asked to accept
    warnings: List[str],       # Tier-1 warnings (logged prominently)
    assume_yes: bool = False,  # Headless mode: accept without prompting
) -> bool:  # True = proceed, False = operator aborted
    """HITL approval seam in its cheapest viable form (log + optional CLI prompt).

    Per-seam HITL-assist annotation (5 fields):
      1. signal: per-document summary + Tier-1 warnings
      2. deterministic pre-filter: tier1_alignment_checks (no AI)
      3. modality-bridge candidate: spectrogram / word-overlay review (future Tier 2)
      4. authoritative verifier: re-align or re-transcribe-and-compare (future Tier 3)
      5. flywheel capture: accept/abort decisions logged; durable capture is a
         pass-2 seam-contract concern, not solved here

    NOTE: input() blocks the event loop — acceptable because seams sit between
    stages with no jobs in flight; the pass-2 seam contract needs an async shape.
    """
    for line in summary_lines:
        logger.info(f"[{seam}] {line}")
    for w in warnings:
        logger.warning(f"[{seam}] {w}")
    if assume_yes:
        logger.info(f"[{seam}] auto-accepted (assume_yes)")
        return True
    reply = input(f"[{seam}] proceed? [Y/n] ").strip().lower()
    accepted = reply in ("", "y", "yes")
    logger.info(f"[{seam}] {'accepted' if accepted else 'ABORTED'} by operator")
    return accepted

In [ ]:
#| export
def collect_plugin_info(
    manager: PluginManager,   # Manager holding the loaded capabilities
    instance_ids: List[str],  # Instance ids to record
) -> Dict[str, Dict[str, Any]]:  # instance_id -> {name, version, db_path}
    """Record capability identity + data-DB pointers for the run manifest (provenance)."""
    info: Dict[str, Dict[str, Any]] = {}
    for iid in instance_ids:
        meta = (getattr(manager, "plugins", {}) or {}).get(iid)
        if meta is None:
            continue
        manifest = getattr(meta, "manifest", {}) or {}
        # Prefer the loaded instance's EFFECTIVE config (a --graph-db-path
        # override must be what downstream cores resolve), falling back to
        # the manifest default.
        inst = (getattr(manager, "instances", {}) or {}).get(iid)
        effective_db = (inst.config or {}).get("db_path") if inst is not None else None
        info[iid] = {"name": meta.name, "version": getattr(meta, "version", None),
                     "db_path": effective_db or manifest.get("db_path")}
    return info

In [ ]:
#| export
async def run_decomp(
    manager: PluginManager,        # Manager with VAD + FA + graph capabilities loaded
    queue: JobQueue,               # Started job queue
    cfg: DecompConfig,             # Run configuration
    source_manifest_path: str,     # Transcription run manifest to decompose
    run_id: Optional[str] = None,  # Override run id (default: generated)
) -> DecompManifest:  # Manifest of the documents produced
    """Decompose every source in a transcription run manifest into graph documents.

    Per source: align -> [alignment-review seam] -> build payload ->
    [commit-review seam] -> commit -> verify. An operator abort at any seam stops
    the run; the manifest holds the documents committed so far.
    """
    run_id = run_id or new_run_id()
    src = load_source_manifest(source_manifest_path)
    transcriber_name = (src.get("config", {}) or {}).get("transcriber_plugin", "unknown")

    manifest = DecompManifest(
        run_id=run_id, created_at=time.time(), config=cfg.to_dict(),
        source_manifest=str(source_manifest_path),
        source_format=src.get("format", ""), source_version=src.get("version", ""),
        plugins=collect_plugin_info(manager, [cfg.vad_plugin, cfg.fa_plugin, cfg.graph_plugin]),
    )

    sources = src.get("sources", []) or []
    for i, source in enumerate(sources):
        source_path, aligned = await decompose_source(
            queue, cfg, source, str(source_manifest_path), i, transcriber_name)
        title = Path(source_path).stem or f"document-{i}"

        empty = sum(1 for a in aligned if not a.text.strip())
        warns = [f"{empty}/{len(aligned)} aligned segment(s) have empty text"] if empty else []
        if not confirm_seam("alignment-review",
                            [f"{title}: {len(aligned)} aligned segment(s)"],
                            warns, assume_yes=cfg.assume_yes):
            logger.warning(f"run {run_id}: aborted at source {i} ({source_path})")
            break

        nodes, edges, doc_id, seg_ids = build_graph_payload(
            title, aligned, str(source_manifest_path), media_type=cfg.media_type)

        if not confirm_seam("commit-review",
                            [f"{title}: committing {len(seg_ids)} segment node(s) + "
                             f"{len(edges)} edge(s) to {cfg.graph_plugin}"],
                            [], assume_yes=cfg.assume_yes):
            logger.warning(f"run {run_id}: commit declined at source {i} ({source_path})")
            break

        counts = await commit_graph(queue, cfg.graph_plugin, nodes, edges)
        logger.info(f"[src {i}] committed {counts['nodes']} node(s), {counts['edges']} edge(s)")

        vr = await verify_document(queue, cfg.graph_plugin, doc_id)
        if vr is None:
            logger.error(f"[src {i}] verify: document {doc_id} NOT FOUND in graph")
        else:
            logger.info(f"[src {i}] verify: {'OK' if vr.ok else 'FAILED'} "
                        f"(segments={vr.segment_count}, starts_with={vr.has_starts_with}, "
                        f"next_chain={vr.next_chain_complete}, part_of={vr.part_of_complete}, "
                        f"timing={vr.all_have_timing}, sources={vr.all_have_sources})")

        manifest.documents.append(DecompDocument(
            document_id=doc_id, source_path=source_path, title=title,
            segment_count=len(seg_ids), segment_ids=seg_ids))
    return manifest

In [ ]:
# pipeline import smoke check (no capabilities involved)
assert callable(run_decomp)
assert callable(decompose_source)
_m = load_source_manifest  # symbol present
print("pipeline import checks OK")

pipeline import checks OK


In [ ]:
# Composition builder + normalizer pure-logic checks (no capabilities involved)
from cjm_media_plugin_system.core import MediaAnalysisResult, TimeRange
from cjm_forced_alignment_adapter_interface.core import ForcedAlignResult, ForcedAlignItem
from cjm_plugin_system.core.ports import new_composition_run

# Normalizers fold typed wire results.
_chunks = vad_chunks_from_result(MediaAnalysisResult(
    ranges=[TimeRange(start=5.0, end=9.0), TimeRange(start=0.5, end=2.0)]))
assert [(c.index, c.start_time) for c in _chunks] == [(0, 0.5), (1, 5.0)]
_words = fa_words_from_result(ForcedAlignResult(
    items=[ForcedAlignItem(text="hi", start_time=0.1, end_time=0.4)]))
assert _words[0].text == "hi" and _words[0].end_time == 0.4

# Whole-source M×(VAD ∥ FA) shape: empty-text psegs skip; all nodes are
# dependency-free (the fan-in happens host-side at the fold).
_segs = [
    {"model_input_path": "/s0.wav", "text": "alpha", "start": 0.0, "job_id": "j0"},
    {"model_input_path": "/s1.wav", "text": "  ", "start": 300.0, "job_id": "j1"},
    {"model_input_path": "/s2.wav", "text": "beta", "start": 600.0, "job_id": "j2"},
]
_comp, _metas = build_alignment_composition(_segs, "silero", "qwen3")
assert len(_comp.nodes) == 4 and len(_metas) == 3
assert _metas[1]["skipped"] is True
assert _comp.nodes[0].kwargs == {"media_path": "/s0.wav", "force": False}
assert _comp.nodes[3].kwargs == {"audio": "/s2.wav", "text": "beta", "force": False}
_run = new_composition_run(_comp, "r")
assert _run.ready_nodes() == ["vad_0000", "fa_0000", "vad_0002", "fa_0002"]
print("alignment composition builder/normalizer checks OK")